Whisper

In [2]:
pip install -U openai-whisper

  Using cached openai_whisper-20250625-py3-none-any.whl
  Using cached more_itertools-11.1.0-py3-none-any.whl (72 kB)
  Using cached numba-0.67.0-cp311-cp311-win_amd64.whl (2.8 MB)
  Using cached tiktoken-0.14.0-cp311-cp311-win_amd64.whl (944 kB)
  Using cached llvmlite-0.49.0-cp311-cp311-win_amd64.whl (41.9 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install torchcodec

  Using cached torchcodec-0.16.0-cp311-cp311-win_amd64.whl (6.4 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Whisper demo

In [ ]:
import os

ffmpeg_bin = os.environ.get("FFMPEG_BIN")
print("FFMPEG_BIN:", ffmpeg_bin)

if ffmpeg_bin:
    os.add_dll_directory(ffmpeg_bin)

FFMPEG_BIN: None


In [1]:
import os

os.environ["FFMPEG_BIN"] = r"G:\ffmpeg\ffmpeg-9.0.1-full_build-shared\bin"

ffmpeg_bin = os.environ.get("FFMPEG_BIN")
print("FFMPEG_BIN:", ffmpeg_bin)

os.add_dll_directory(ffmpeg_bin)


FFMPEG_BIN: G:\ffmpeg\ffmpeg-9.0.1-full_build-shared\bin


<AddedDllDirectory('G:\\ffmpeg\\ffmpeg-9.0.1-full_build-shared\\bin')>

In [2]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset
import torchcodec
import whisper
from datasets import Dataset, Audio
from pathlib import Path

g:\Uni\Sem2\NLP\term_paper\whisper\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# --------------------------------------------------
# Model
# --------------------------------------------------

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-base.en"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    
)

model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

dataset = Path(r"G:\Uni\Sem2\NLP\term_paper\data\librispeech\LibriSpeech\test-clean\61\70968\61-70968-0000.flac")


Loading weights: 100%|██████████| 245/245 [00:00<00:00, 4776.58it/s]


In [ ]:
def transcribe_nbest(audio_file, n):

    # Load and resample audio using the existing HF processor
    audio = Audio(sampling_rate=16000).decode_example({
    "path": str(audio_file),
    "bytes": None,
})
    inputs = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt",
    )

    input_features = inputs.input_features.to(
        device=device,
        dtype=torch_dtype,
    )

    generated = model.generate(
        input_features,
        num_beams=n,
        num_return_sequences=n,
        diversity_penalty=1.0,
        do_sample=True,
        return_dict_in_generate=True,
        output_scores=True,
        temperature = 0.5
    )

    hypotheses = processor.batch_decode(
    generated.sequences,
    skip_special_tokens=True,
)

    for i, h in enumerate(hypotheses, 1):
        print(f"{i}: {h}")

    print("\nUnique hypotheses:", len(set(hypotheses)))
    print("\nScores:")
    for score in generated.sequences_scores:
        print(score.item())

    return hypotheses

In [22]:
def transcribe_nbest(audio_file, n):

    # Load and resample audio
    audio = Audio(sampling_rate=16000).decode_example({
        "path": str(audio_file),
        "bytes": None,
    })

    inputs = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt",
    )

    input_features = inputs.input_features.to(
        device=device,
        dtype=torch_dtype,
    )

    generated = model.generate(
        input_features,
        do_sample=False,
        temperature=0.5,
        num_beams = n,
        num_return_sequences=n,
        return_dict_in_generate=True,
        output_scores=True,
    )

    hypotheses = processor.batch_decode(
        generated.sequences,
        skip_special_tokens=True,
    )

    for i, h in enumerate(hypotheses, 1):
        print(f"{i}: {h}")

    print("\nUnique hypotheses:", len(set(hypotheses)))

    print("\nScores:")
    for score in generated.sequences_scores:
        print(score.item())


    return hypotheses


In [23]:
audio = dataset

hypotheses = transcribe_nbest(
    audio,
    n=10,
)

1:  He began a confused complaint against the wizard, who had vanished behind the curtain on the left.
2:  He began a confused complaint against the wizard, who had vanished behind the curtain on the left.
3:  He began a confused complaint against the wizard who had vanished behind the curtain on the left.
4:  He began a confused complaint against the wizard, who had vanished behind the curtain on the left.
5:  He began a confused complaint against the wizard, who had vanished behind the curtain on the left.
6:  He began a confused complaint against the wizard who had vanished behind the curtain on the left.
7:  He began a confused complaint against the wizard who had vanished behind the curtain on the left.
8:  He began a confused complaint against the wizard who had vanished behind the curtain on the left.
9:  He began a confused complaint against the wizard, who had vanished behind the curtain on the left.
10:  He began a confused complaint against the wizard who had vanished behind

AttributeError: 'GenerateEncoderDecoderOutput' object has no attribute 'sequences_scores'

*Note - Whisper ASR resulted in identical hypotheses - testing other alternatives to simulate N-best?

In [20]:
dataset_path = Path(r"G:\Uni\Sem2\NLP\term_paper\librispeech\LibriSpeech\test-clean\61\70968")

audio_extensions = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}

audio_files = [
    str(file)
    for file in dataset_path.rglob("*")
    if file.is_file() and file.suffix.lower() in audio_extensions
]

In [23]:
audio = audio_files[0]

hypotheses = transcribe_nbest(
    audio,
    n=5,
)

for i, hypothesis in enumerate(hypotheses, start=1):
    print(f"{i}: {hypothesis}")

TypeError: string indices must be integers, not 'str'

In [30]:
print(dataset)
print(dataset.features)
print(type(dataset[0]))
print(dataset[0])

G:\Uni\Sem2\NLP\term_paper\librispeech\LibriSpeech\test-clean\61\70968\61-70968-0000


AttributeError: 'WindowsPath' object has no attribute 'features'

In [ ]:
'''
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    chunk_length_s=30,
)

# --------------------------------------------------
# Dataset
# --------------------------------------------------

dataset_path = Path(r"G:\Uni\Sem2\NLP\term_paper\librispeech\LibriSpeech\test-clean")

audio_extensions = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}

audio_files = [
    str(file)
    for file in dataset_path.rglob("*")
    if file.is_file() and file.suffix.lower() in audio_extensions
]

audio_files.sort()

dataset = Dataset.from_dict({
    "audio": audio_files,
})

# Tell Hugging Face Datasets to decode the audio
dataset = dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000),
)

print(f"Found {len(dataset)} audio files.")

# --------------------------------------------------
# Inference
# --------------------------------------------------

results = []

batch_size = 5

for start in range(0, len(audio_files), batch_size):
    batch_files = audio_files[start:start + batch_size]

    print(
        f"Transcribing {start + 1}-"
        f"{min(start + batch_size, len(audio_files))}/{len(audio_files)}..."
    )

    batch_results = pipe(
        batch_files,
        return_timestamps=True,
        batch_size=batch_size,
    )

    results.extend([result["text"] for result in batch_results])

print("Done!")


# --------------------------------------------------
# Save results
# --------------------------------------------------

with open("transcriptions.txt", "w", encoding="utf-8") as f:
    for audio_file, transcription in zip(audio_files, results):
        f.write(f"{os.path.basename(audio_file)}\t{transcription}\n")
'''


NeMo Conformer demo

In [1]:
import sys
import torch
import nemo.collections.asr as nemo_asr
from pathlib import Path
import json
import soundfile as sf

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("NeMo:", nemo_asr.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

g:\Uni\Sem2\NLP\term_paper\nemo_312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
PyTorch: 2.6.0+cu124
CUDA: 12.4
CUDA available: True
NeMo: 3.0.0
GPU: NVIDIA GeForce GTX 1650 SUPER


File Loading

In [2]:
LIBRISPEECH = Path(r"G:\Uni\Sem2\NLP\term_paper\data\librispeech\LibriSpeech")

print(LIBRISPEECH.exists())
print(list(LIBRISPEECH.iterdir())[:10])

wav_files = list((LIBRISPEECH / "test-clean").rglob("*.flac"))

print("Number of audio files:", len(wav_files))
print("First file:", wav_files[0])


subset = LIBRISPEECH / "test-clean"

transcripts = {}

for trans_file in subset.rglob("*.trans.txt"):
    with open(trans_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            utterance_id, text = line.split(" ", 1)
            transcripts[utterance_id] = text.lower()

print("Transcripts:", len(transcripts))
print(next(iter(transcripts.items())))


manifest_path = Path("data/librispeech_test_clean.json")

count = 0

with open(manifest_path, "w", encoding="utf-8") as f:
    for wav in wav_files:
        # LibriSpeech file names are like:
        # 19-198-0000.flac
        utterance_id = wav.stem

        if utterance_id not in transcripts:
            continue

        info = sf.info(wav)

        entry = {
            "audio_filepath": str(wav.resolve()),
            "duration": info.duration,
            "text": transcripts[utterance_id],
        }

        f.write(json.dumps(entry) + "\n")
        count += 1

print(f"Wrote {count} entries to {manifest_path}")

True
[WindowsPath('G:/Uni/Sem2/NLP/term_paper/data/librispeech/LibriSpeech/BOOKS.TXT'), WindowsPath('G:/Uni/Sem2/NLP/term_paper/data/librispeech/LibriSpeech/CHAPTERS.TXT'), WindowsPath('G:/Uni/Sem2/NLP/term_paper/data/librispeech/LibriSpeech/dev-clean'), WindowsPath('G:/Uni/Sem2/NLP/term_paper/data/librispeech/LibriSpeech/LICENSE.TXT'), WindowsPath('G:/Uni/Sem2/NLP/term_paper/data/librispeech/LibriSpeech/README.TXT'), WindowsPath('G:/Uni/Sem2/NLP/term_paper/data/librispeech/LibriSpeech/SPEAKERS.TXT'), WindowsPath('G:/Uni/Sem2/NLP/term_paper/data/librispeech/LibriSpeech/test-clean'), WindowsPath('G:/Uni/Sem2/NLP/term_paper/data/librispeech/LibriSpeech/train-clean-100')]
Number of audio files: 2620
First file: G:\Uni\Sem2\NLP\term_paper\data\librispeech\LibriSpeech\test-clean\1089\134686\1089-134686-0000.flac
Transcripts: 2620
('1089-134686-0000', 'he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flo

In [3]:
from nemo.collections.asr.models import EncDecRNNTBPEModel
model = nemo_asr.models.EncDecRNNTBPEModel.from_pretrained(
    model_name="stt_en_conformer_transducer_small"
)
print("Model loaded!")

[NeMo I 2026-09-03 10:45:58 cloud:58] Found existing object C:\Users\Bao Long\.cache\torch\NeMo\NeMo_3.0.0\stt_en_conformer_transducer_small\a755afe69952642a8410330876938b83\stt_en_conformer_transducer_small.nemo.
[NeMo I 2026-09-03 10:45:58 cloud:64] Re-using file from: C:\Users\Bao Long\.cache\torch\NeMo\NeMo_3.0.0\stt_en_conformer_transducer_small\a755afe69952642a8410330876938b83\stt_en_conformer_transducer_small.nemo
[NeMo I 2026-09-03 10:45:58 common:1122] Instantiating model from pre-trained checkpoint
[NeMo I 2026-09-03 10:46:00 mixins:194] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-09-03 10:46:00 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /data2/nemo_asr_set_2.0/RES/tarred_audio_manifest.json
    sample_rate: 16000
    batch_size: 16
    shuffle: true
    num_workers: 4
    pin_memory: true
    use_start_end_token: true
    trim_silence: false
    max_duration: 20
    min_duration: 0.1
    shuffle_n: 2048
    is_tarred: true
    tarred_audio_filepaths: /data2/nemo_asr_set_2.0/RES/audio__OP_0..4095_CL_.tar
    
[NeMo W 2026-09-03 10:46:00 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath:
    - /manifests/librispeech/librivox-dev-other.json
    - /manifest

[NeMo I 2026-09-03 10:46:01 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0}
[NeMo I 2026-09-03 10:46:01 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0}
[NeMo I 2026-09-03 10:46:01 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0}
[NeMo I 2026-09-03 10:46:02 save_restore_connector:287] Model EncDecRNNTBPEModel was successfully restored from C:\Users\Bao Long\.cache\torch\NeMo\NeMo_3.0.0\stt_en_conformer_transducer_small\a755afe69952642a8410330876938b83\stt_en_conformer_transducer_small.nemo.
Model loaded!


In [ ]:
from omegaconf import OmegaConf
print(OmegaConf.to_yaml(model.cfg.decoding))

strategy: greedy_batch
greedy:
  max_symbols: 5
beam:
  beam_size: 2
  return_best_hypothesis: false
  score_norm: true
  tsd_max_sym_exp: 50
  alsd_max_target_len: 2.0



In [9]:
decoding_cfg = model.cfg.decoding

decoding_cfg.strategy = "beam"
decoding_cfg.beam.beam_size = 20
decoding_cfg.beam.search_type = "default"
decoding_cfg.beam.return_best_hypothesis = False
decoding_cfg.beam.score_norm = True
decoding_cfg.beam.softmax_temperature = 1.2
model.change_decoding_strategy(decoding_cfg)

[NeMo I 2026-09-03 10:47:00 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0}


[NeMo W 2026-09-03 10:47:00 rnnt_decoding:506] Decoding strategy `beam` is experimental. Recommended beam decoding strategy is `malsd_batch`.


[NeMo I 2026-09-03 10:47:00 rnnt_bpe_models:436] Changed decoding strategy to 
    model_type: rnnt
    strategy: beam
    compute_hypothesis_token_set: false
    preserve_alignments: null
    tdt_include_token_duration: null
    confidence_cfg:
      preserve_frame_confidence: false
      preserve_token_confidence: false
      preserve_word_confidence: false
      exclude_blank: true
      aggregation: min
      tdt_include_duration: false
      method_cfg:
        name: entropy
        entropy_type: tsallis
        alpha: 0.33
        entropy_norm: exp
        temperature: DEPRECATED
    fused_batch_size: null
    compute_timestamps: null
    compute_langs: false
    word_seperator: ' '
    segment_seperators:
    - .
    - '!'
    - '?'
    segment_gap_threshold: null
    rnnt_timestamp_type: all
    greedy:
      max_symbols_per_step: 10
      preserve_alignments: false
      preserve_frame_confidence: false
      tdt_include_token_duration: false
      tdt_include_duration_confide

Inference

In [9]:
audio_file = str(wav_files[0])

print(audio_file)
output = model.transcribe([audio_file])

[NeMo W 2026-09-01 17:17:14 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-09-01 17:17:14 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


G:\Uni\Sem2\NLP\term_paper\data\librispeech\LibriSpeech\test-clean\1089\134686\1089-134686-0000.flac


Beam search progress:: 100%|██████████| 1/1 [00:40<00:00, 40.67s/sample]
Transcribing: 1it [00:40, 40.75s/it]


WER

In [ ]:
batch_size = 32

all_references = []
all_hypotheses = []

for start in range(0, len(wav_files), batch_size):

    batch_wavs = wav_files[start:start + batch_size]

    audio_batch = []
    references = []

    for wav in batch_wavs:
        utterance_id = wav.stem

        if utterance_id not in transcripts:
            continue

        audio_batch.append(str(wav))
        references.append(transcripts[utterance_id])

    if not audio_batch:
        continue

    hypotheses = model.transcribe(audio_batch)

    hypotheses_text = [
        h.text if hasattr(h, "text") else str(h)
        for h in hypotheses
    ]

    all_references.extend(references)
    all_hypotheses.extend(hypotheses_text)

    print(f"Processed {len(all_references)} files")

score = wer(all_references, all_hypotheses)

print(f"Files: {len(all_references)}")
print(f"Corpus WER: {score:.2%}")


In [17]:
import time
import json
from pathlib import Path
from collections import defaultdict
from jiwer import wer

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

SUBSET_SIZE = 10
BATCH_SIZE = 2
NUM_HYPOTHESES = 20

subset = LIBRISPEECH / "test-clean"

# ---------------------------------------------------------
# Load LibriSpeech references
# ---------------------------------------------------------

transcripts = {}

for trans_file in subset.rglob("*.trans.txt"):

    with open(trans_file, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            utterance_id, text = line.split(" ", 1)

            transcripts[utterance_id] = text.lower()

# ---------------------------------------------------------
# Find audio files
# ---------------------------------------------------------

wav_files = sorted(
    subset.rglob("*.flac")
)

print(
    f"Total audio files: {len(wav_files)}"
)

# ---------------------------------------------------------
# Select subset
# ---------------------------------------------------------

selected_wavs = []

for wav in wav_files:

    utterance_id = wav.stem

    if utterance_id in transcripts:

        selected_wavs.append(wav)

    if len(selected_wavs) >= SUBSET_SIZE:
        break

print(
    f"Selected files: {len(selected_wavs)}"
)

print(
    f"First file: {selected_wavs[0]}"
)

print(
    f"Last file:  {selected_wavs[-1]}"
)

# ---------------------------------------------------------
# Run ASR
# ---------------------------------------------------------

all_results = []

start_time = time.time()

for batch_start in range(
    0,
    len(selected_wavs),
    BATCH_SIZE
):

    batch_wavs = selected_wavs[
        batch_start:
        batch_start + BATCH_SIZE
    ]

    # -----------------------------------------------------
    # Prepare batch
    # -----------------------------------------------------

    audio_batch = []
    references = []
    utterance_ids = []

    for wav in batch_wavs:

        utterance_id = wav.stem

        audio_batch.append(str(wav))
        references.append(transcripts[utterance_id])
        utterance_ids.append(utterance_id)

    print(
        f"\nProcessing files "
        f"{batch_start + 1}-"
        f"{batch_start + len(audio_batch)} "
        f"of {len(selected_wavs)}"
    )

    batch_start_time = time.time()

    # -----------------------------------------------------
    # Transcribe
    # -----------------------------------------------------

    hypotheses = model.transcribe(
        audio_batch,
        batch_size=BATCH_SIZE,
    )

    # -----------------------------------------------------
    # Verify batch output
    # -----------------------------------------------------

    if len(hypotheses) != len(audio_batch):

        raise RuntimeError(
            f"Number of hypothesis groups "
            f"({len(hypotheses)}) does not match "
            f"number of audio files "
            f"({len(audio_batch)})."
        )

    # -----------------------------------------------------
    # Process N-best hypotheses
    # -----------------------------------------------------

    for utterance_id, reference, h_list in zip(
        utterance_ids,
        references,
        hypotheses,
    ):

        print(f"\nID:         {utterance_id}")
        print(f"Reference:  {reference}")

        # -------------------------------------------------
        # Create result for this utterance
        # -------------------------------------------------

        utterance_result = {
            "utterance_id": utterance_id,
            "reference": reference,
            "hypotheses": []
        }

        # -------------------------------------------------
        # Verify N-best size
        # -------------------------------------------------

        if len(h_list) != NUM_HYPOTHESES:

            raise RuntimeError(
                f"{utterance_id}: expected "
                f"{NUM_HYPOTHESES} hypotheses, "
                f"got {len(h_list)}"
            )

        # -------------------------------------------------
        # Process all hypotheses
        # -------------------------------------------------

        for rank, h in enumerate(
            h_list,
            start=1
        ):

            if not hasattr(h, "text"):

                raise TypeError(
                    f"Unexpected hypothesis type: "
                    f"{type(h)}"
                )

            hypothesis_text = h.text.strip()

            # NeMo score
            hypothesis_score = float(h.score)

            # WER against reference
            hypothesis_wer = wer(
                reference,
                hypothesis_text
            )

            print(
                f"\nHypothesis {rank:2d}: "
                f"{hypothesis_text}"
            )

            print(
                f"Score:           "
                f"{hypothesis_score:.6f}"
            )

            print(
                f"WER:             "
                f"{hypothesis_wer:.2%}"
            )

            # -------------------------------------------------
            # Save hypothesis
            # -------------------------------------------------

            utterance_result["hypotheses"].append({
                "rank": rank,
                "score": hypothesis_score,
                "text": hypothesis_text,
                "wer": hypothesis_wer,
            })

        # -------------------------------------------------
        # IMPORTANT:
        # Append AFTER all 20 hypotheses are processed
        # -------------------------------------------------

        all_results.append(
            utterance_result
        )

    # -----------------------------------------------------
    # Batch timing
    # -----------------------------------------------------

    batch_time = time.time() - batch_start_time

    print(
        f"\nBatch time: "
        f"{batch_time / 60:.2f} min"
    )


    # -----------------------------------------------------
    # Timing
    # -----------------------------------------------------

    batch_time = (
        time.time() - batch_start_time
    )

    print(
        f"\nBatch time: "
        f"{batch_time / 60:.2f} min"
    )

# =========================================================
# N-BEST DIVERSITY STATISTICS
# =========================================================

print("\n" + "=" * 60)
print(f"N-BEST DIVERSITY @ {NUM_HYPOTHESES}")
print("=" * 60)

diversity_results = []

for result in all_results:

    # -----------------------------------------------------
    # Get the 20 hypothesis texts
    # -----------------------------------------------------

    texts = [
        h["text"]
        for h in result["hypotheses"]
    ]

    # -----------------------------------------------------
    # Count unique hypotheses
    # -----------------------------------------------------

    total = len(texts)
    unique = len(set(texts))

    diversity = unique / total if total > 0 else 0.0

    # -----------------------------------------------------
    # Save result
    # -----------------------------------------------------

    diversity_results.append({
        "utterance_id": result["utterance_id"],
        "total_hypotheses": total,
        "unique_hypotheses": unique,
        "diversity": diversity
    })

    # -----------------------------------------------------
    # Print result
    # -----------------------------------------------------

    print(
        f"{result['utterance_id']}: "
        f"{unique}/{total} unique "
        f"({diversity:.2%})"
    )


# =========================================================
# CORPUS-LEVEL DIVERSITY
# =========================================================

diversity_values = [
    r["diversity"]
    for r in diversity_results
]

unique_counts = [
    r["unique_hypotheses"]
    for r in diversity_results
]

mean_diversity = (
    sum(diversity_values) / len(diversity_values)
)

mean_unique = (
    sum(unique_counts) / len(unique_counts)
)

fully_duplicate = sum(
    1
    for r in diversity_results
    if r["unique_hypotheses"] == 1
)

print("\n" + "-" * 60)
print("CORPUS DIVERSITY")
print("-" * 60)

print(
    f"Mean unique hypotheses: "
    f"{mean_unique:.2f} / {NUM_HYPOTHESES}"
)

print(
    f"Mean diversity ratio:   "
    f"{mean_diversity:.2%}"
)

print(
    f"Minimum diversity:      "
    f"{min(diversity_values):.2%}"
)

print(
    f"Maximum diversity:      "
    f"{max(diversity_values):.2%}"
)

print(
    f"Fully duplicated lists: "
    f"{fully_duplicate}/{len(all_results)}"
)

print(
    f"Fully duplicated ratio: "
    f"{fully_duplicate / len(all_results):.2%}"
)

# =========================================================
# CORPUS-LEVEL WER BY RANK
# =========================================================

print(
    "\n" + "=" * 60
)

print(
    "CORPUS WER BY HYPOTHESIS RANK"
)

print(
    "=" * 60
)

corpus_wer_by_rank = {}

for rank in range(
    1,
    NUM_HYPOTHESES + 1
):

    rank_references = []
    rank_hypotheses = []

    # -----------------------------------------------------
    # Get the hypothesis at this rank from every utterance
    # -----------------------------------------------------

    for result in all_results:

        rank_hypothesis = result["hypotheses"][
            rank - 1
        ]

        rank_references.append(
            result["reference"]
        )

        rank_hypotheses.append(
            rank_hypothesis["text"]
        )

    # -----------------------------------------------------
    # Calculate corpus WER for this rank
    # -----------------------------------------------------

    rank_wer = wer(
        rank_references,
        rank_hypotheses
    )

    corpus_wer_by_rank[rank] = rank_wer

    print(
        f"Hypothesis {rank:2d}: "
        f"{rank_wer:.2%}"
    )


# =========================================================
# ORACLE WER
# =========================================================

print(
    "\n" + "=" * 60
)

print(
    "ORACLE WER"
)

print(
    "=" * 60
)

oracle_results = []

# ---------------------------------------------------------
# Find best hypothesis for each utterance
# ---------------------------------------------------------

for result in all_results:

    best_hypothesis = min(
        result["hypotheses"],
        key=lambda h: h["wer"]
    )

    oracle_results.append({
        "utterance_id": result["utterance_id"],
        "reference": result["reference"],
        "rank": best_hypothesis["rank"],
        "score": best_hypothesis["score"],
        "text": best_hypothesis["text"],
        "wer": best_hypothesis["wer"],
    })

    print(
        f"{result['utterance_id']}: "
        f"best rank = "
        f"{best_hypothesis['rank']:2d}, "
        f"WER = "
        f"{best_hypothesis['wer']:.2%}"
    )

# ---------------------------------------------------------
# Mean oracle WER
# ---------------------------------------------------------

oracle_wers = [
    result["wer"]
    for result in oracle_results
]

oracle_mean_wer = (
    sum(oracle_wers)
    / len(oracle_wers)
)

print(
    f"\nOracle WER@{NUM_HYPOTHESES}: "
    f"{oracle_mean_wer:.2%}"
)


# =========================================================
# TIMING
# =========================================================

total_time = (
    time.time() - start_time
)

# =========================================================
# GENERAL RESULTS
# =========================================================

print(
    "\n" + "=" * 60
)

print(
    "RESULTS"
)

print(
    "=" * 60
)

print(
    f"Files:        "
    f"{len(selected_wavs)}"
)

print(
    f"Hypotheses:   "
    f"{NUM_HYPOTHESES} per file"
)

print(
    f"Total results:"
    f" {len(all_results)}"
)

print(
    f"Top-1 WER:    "
    f"{corpus_wer_by_rank[1]:.2%}"
)

print(
    f"Oracle WER@20:"
    f" {oracle_mean_wer:.2%}"
)

print(
    f"Total time:   "
    f"{total_time / 60:.2f} minutes"
)

print(
    f"Time/file:    "
    f"{total_time / len(selected_wavs):.2f} seconds"
)

print(
    f"Files/hour:   "
    f"{len(selected_wavs) / (total_time / 3600):.1f}"
)


# =========================================================
# WER STATISTICS
# =========================================================

print(
    "\nWER statistics "
    "(all hypothesis results):"
)

individual_wers = [
    hypothesis["wer"]
    for result in all_results
    for hypothesis in result["hypotheses"]
]

print(
    f"Minimum:      "
    f"{min(individual_wers):.2%}"
)

print(
    f"Maximum:      "
    f"{max(individual_wers):.2%}"
)

print(
    f"Mean:         "
    f"{sum(individual_wers) / len(individual_wers):.2%}"
)


# =========================================================
# UNIQUE N-BEST / ORACLE ANALYSIS
# =========================================================

print(
    "\n" + "=" * 60
)

print(
    "UNIQUE N-BEST ANALYSIS"
)

print(
    "=" * 60
)

unique_oracle_wers = []

for result in all_results:

    # Keep only the first occurrence of each text
    seen = set()
    unique_hypotheses = []

    for h in result["hypotheses"]:

        if h["text"] not in seen:
            seen.add(h["text"])
            unique_hypotheses.append(h)

    # Best WER among unique hypotheses
    best_unique = min(
        unique_hypotheses,
        key=lambda h: h["wer"]
    )

    unique_oracle_wers.append(
        best_unique["wer"]
    )

    print(
        f"{result['utterance_id']}: "
        f"{len(unique_hypotheses)} unique / "
        f"{NUM_HYPOTHESES} total, "
        f"unique oracle WER = "
        f"{best_unique['wer']:.2%}"
    )

unique_oracle_wer = (
    sum(unique_oracle_wers)
    / len(unique_oracle_wers)
)

print(
    f"\nUnique Oracle WER@{NUM_HYPOTHESES}: "
    f"{unique_oracle_wer:.2%}"
)

# =========================================================
# SAVE RESULTS
# =========================================================

output_path = Path(
    f"results_librispeech_"
    f"{SUBSET_SIZE}_"
    f"{NUM_HYPOTHESES}best.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    f"\nSaved results to: {output_path}"
)



[NeMo W 2026-09-03 11:25:59 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-09-03 11:25:59 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


Total audio files: 2620
Selected files: 10
First file: G:\Uni\Sem2\NLP\term_paper\data\librispeech\LibriSpeech\test-clean\1089\134686\1089-134686-0000.flac
Last file:  G:\Uni\Sem2\NLP\term_paper\data\librispeech\LibriSpeech\test-clean\1089\134686\1089-134686-0009.flac

Processing files 1-2 of 10


Beam search progress:: 100%|██████████| 2/2 [01:05<00:00, 32.53s/sample]
Transcribing: 1it [01:05, 65.13s/it]
[NeMo W 2026-09-03 11:27:04 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-09-03 11:27:04 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)



ID:         1089-134686-0000
Reference:  he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce

Hypothesis  1: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce
Score:           -22.535836
WER:             0.00%

Hypothesis  2: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce
Score:           -22.605088
WER:             0.00%

Hypothesis  3: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce
Score:           -22.606688
WER:             0.00%

Hypothesis  4: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to

Beam search progress:: 100%|██████████| 2/2 [00:37<00:00, 18.91s/sample]
Transcribing: 1it [00:38, 38.33s/it]
[NeMo W 2026-09-03 11:27:42 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-09-03 11:27:42 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


...


ID:         1089-134686-0004
Reference:  number ten fresh nelly is waiting on you good night husband

Hypothesis  1: number ten fresh nellie is waiting on you good night husband
Score:           -7.813086
WER:             9.09%

Hypothesis  2: number ten fresh nellie is waiting on you good night husband
Score:           -7.845376
WER:             9.09%

Hypothesis  3: number ten fresh nellie is waiting on you good night husband
Score:           -7.892435
WER:             9.09%

Hypothesis  4: number ten fresh nellie is waiting on you good night husband
Score:           -7.924725
WER:             9.09%

Hypothesis  5: number ten fresh nellie is waiting on you good night husband
Score:           -8.044255
WER:             9.09%

Hypothesis  6: number ten fresh nellie is waiting on you good night husband
Score:           -8.051879
WER:             9.09%

Hypothesis  7: number ten fresh nellie is waiting on you good night husband
Score:           -8.076545
WER:             9.09%

Hypothes

Beam search progress:: 100%|██████████| 2/2 [01:52<00:00, 56.23s/sample]
Transcribing: 1it [01:52, 112.54s/it]
[NeMo W 2026-09-03 11:31:53 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-09-03 11:31:53 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)



ID:         1089-134686-0006
Reference:  the dull light fell more faintly upon the page whereon another equation began to unfold itself slowly and to spread abroad its widening tail

Hypothesis  1: the dull light fell more faintly upon the page whereon another equation began to unfold itself slowly and to spread abroad its widening tail
Score:           -19.253956
WER:             0.00%

Hypothesis  2: the dull light fell more faintly upon the page whereon another equation began to unfold itself slowly and to spread abroad its widening tail
Score:           -19.293965
WER:             0.00%

Hypothesis  3: the dull light fell more faintly upon the page whereon another equation began to unfold itself slowly and to spread abroad its widening tail
Score:           -19.296299
WER:             0.00%

Hypothesis  4: the dull light fell more faintly upon the page whereon another equation began to unfold itself slowly and to spread abroad its widening tail
Score:           -19.329809
WER:    

Beam search progress:: 100%|██████████| 2/2 [01:19<00:00, 39.63s/sample]
Transcribing: 1it [01:19, 79.42s/it]


ID:         1089-134686-0008
Reference:  the chaos in which his ardour extinguished itself was a cold indifferent knowledge of himself

Hypothesis  1: the chaos in which his ardour extinguished itself was a cold indifferent knowledge of himself
Score:           -14.758720
WER:             0.00%

Hypothesis  2: the chaos in which his ardour extinguished itself was a cold indifferent knowledge of himself
Score:           -14.762439
WER:             0.00%

Hypothesis  3: the chaos in which his ardour extinguished itself was a cold indifferent knowledge of himself
Score:           -14.796633
WER:             0.00%

Hypothesis  4: the chaos in which his ardour extinguished itself was a cold indifferent knowledge of himself
Score:           -14.800351
WER:             0.00%

Hypothesis  5: the chaos in which his ardour extinguished itself was a cold indifferent knowledge of himself
Score:           -14.861333
WER:             0.00%

Hypothesis  6: the chaos in which his ardour extinguished 

T5

In [18]:
!pip install -q transformers sentencepiece accelerate

In [19]:
import json
import torch
import numpy as np
from pathlib import Path
from transformers import T5Tokenizer, T5ForConditionalGeneration
from jiwer import wer

In [21]:
RESULTS_FILE = Path(
    r"G:\Uni\Sem2\NLP\term_paper\results_librispeech_10_20best.json"
)

with open(RESULTS_FILE, "r", encoding="utf-8") as f:
    all_results = json.load(f)

print(f"Loaded {len(all_results)} utterances")
print(all_results[0])

Loaded 10 utterances
{'utterance_id': '1089-134686-0000', 'reference': 'he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce', 'hypotheses': [{'rank': 1, 'score': -22.535836407553617, 'text': 'he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce', 'wer': 0.0}, {'rank': 2, 'score': -22.60508780331172, 'text': 'he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce', 'wer': 0.0}, {'rank': 3, 'score': -22.60668849281403, 'text': 'he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce', 'wer': 0.0}, {'rank': 4, 'score': -22.645618363842843, 'text': 'he hoped there would be stew

In [22]:
MODEL_NAME = "t5-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
t5_model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

t5_model.to(device)
t5_model.eval()

Using device: cuda


[NeMo W 2026-09-03 11:59:33 nemo_logging:364] g:\Uni\Sem2\NLP\term_paper\nemo_312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in G:\.cache\huggingface\hub\models--t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
    To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
      warnings.warn(message)
    
Loading weights: 100%|██████████| 257/257 [00:00<00:00, 7341.29it/s]


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [23]:
print(f"Model: {MODEL_NAME}")
print(f"Device: {device}")

Model: t5-base
Device: cuda


In [24]:
def build_t5_input(hypotheses, top_k=5):
    """
    Convert an N-best ASR list into a T5 input sequence.
    """

    selected = hypotheses[:top_k]

    lines = []

    for h in selected:
        lines.append(f"hypothesis {h['rank']}: {h['text']}")

    input_text = (
        "correct the transcription:\n"
        + "\n".join(lines)
    )

    return input_text

In [25]:
example_input = build_t5_input(
    all_results[0]["hypotheses"],
    top_k=5
)

print(example_input)

correct the transcription:
hypothesis 1: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce
hypothesis 2: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce
hypothesis 3: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce
hypothesis 4: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce
hypothesis 5: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce


In [26]:
def t5_correct(input_text, max_input_length=512, max_output_length=128):
    """
    Run T5 on one ASR N-best input.
    """

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length,
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = t5_model.generate(
            **inputs,
            max_new_tokens=max_output_length,
            num_beams=4,
            early_stopping=True,
        )

    corrected_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return corrected_text.strip()

In [27]:
result = all_results[0]

t5_input = build_t5_input(
    result["hypotheses"],
    top_k=5
)

print("===== T5 INPUT =====")
print(t5_input)

corrected = t5_correct(t5_input)

print("\n===== T5 OUTPUT =====")
print(corrected)

print("\n===== REFERENCE =====")
print(result["reference"])

print("\n===== WER =====")
print(wer(result["reference"], corrected))

===== T5 INPUT =====
correct the transcription:
hypothesis 1: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce
hypothesis 2: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce
hypothesis 3: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce
hypothesis 4: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce
hypothesis 5: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce

===== T5 OUTPUT =====
not_equivalent

===== REFERENCE =====
he hoped there would be st

In [28]:
TOP_K = 5

t5_results = []

for i, result in enumerate(all_results):

    hypotheses = result["hypotheses"]

    t5_input = build_t5_input(
        hypotheses,
        top_k=TOP_K
    )

    corrected_text = t5_correct(t5_input)

    reference = result["reference"]

    corrected_wer = wer(
        reference,
        corrected_text
    )

    t5_result = {
        "utterance_id": result["utterance_id"],
        "reference": reference,
        "top_k": TOP_K,
        "t5_input": t5_input,
        "corrected_text": corrected_text,
        "wer": corrected_wer,
    }

    t5_results.append(t5_result)

    print(
        f"[{i+1}/{len(all_results)}] "
        f"{result['utterance_id']} | "
        f"WER: {corrected_wer:.3f}"
    )

[1/10] 1089-134686-0000 | WER: 1.000
[2/10] 1089-134686-0001 | WER: 1.000
[3/10] 1089-134686-0002 | WER: 1.000
[4/10] 1089-134686-0003 | WER: 1.000
[5/10] 1089-134686-0004 | WER: 1.000
[6/10] 1089-134686-0005 | WER: 1.000
[7/10] 1089-134686-0006 | WER: 1.000
[8/10] 1089-134686-0007 | WER: 1.000
[9/10] 1089-134686-0008 | WER: 1.000
[10/10] 1089-134686-0009 | WER: 1.000


In [29]:
references = [
    result["reference"]
    for result in t5_results
]

predictions = [
    result["corrected_text"]
    for result in t5_results
]

t5_corpus_wer = wer(
    references,
    predictions
)

print(f"T5 corpus WER: {t5_corpus_wer:.2%}")

T5 corpus WER: 100.00%


In [30]:
asr_1best_wer = wer(
    [r["reference"] for r in all_results],
    [r["hypotheses"][0]["text"] for r in all_results]
)

print(f"ASR 1-best WER: {asr_1best_wer:.2%}")

ASR 1-best WER: 1.19%


In [31]:
oracle_predictions = []

for result in all_results:
    best_hypothesis = min(
        result["hypotheses"],
        key=lambda h: h["wer"]
    )

    oracle_predictions.append(
        best_hypothesis["text"]
    )

oracle_wer = wer(
    [r["reference"] for r in all_results],
    oracle_predictions
)

print(f"N-best oracle WER: {oracle_wer:.2%}")

N-best oracle WER: 0.00%


In [32]:
print(f"T5 WER: {t5_corpus_wer:.2%}")

T5 WER: 100.00%


Different N-best sizes

In [33]:
TOP_K_VALUES = [1, 5, 10, 20]

all_t5_results = {}

for top_k in TOP_K_VALUES:

    print(f"\n{'='*50}")
    print(f"Running T5 with top_k = {top_k}")
    print(f"{'='*50}")

    results = []

    for i, result in enumerate(all_results):

        t5_input = build_t5_input(
            result["hypotheses"],
            top_k=top_k
        )

        corrected_text = t5_correct(t5_input)

        results.append({
            "utterance_id": result["utterance_id"],
            "reference": result["reference"],
            "corrected_text": corrected_text,
            "wer": wer(
                result["reference"],
                corrected_text
            )
        })

        print(
            f"[{i+1}/{len(all_results)}]",
            end="\r"
        )

    all_t5_results[top_k] = results

    corpus_wer = wer(
        [r["reference"] for r in results],
        [r["corrected_text"] for r in results]
    )

    print(f"\nTop-{top_k} T5 WER: {corpus_wer:.2%}")


Running T5 with top_k = 1
[10/10]
Top-1 T5 WER: 100.00%

Running T5 with top_k = 5
[10/10]
Top-5 T5 WER: 100.00%

Running T5 with top_k = 10
[10/10]
Top-10 T5 WER: 100.00%

Running T5 with top_k = 20
[10/10]
Top-20 T5 WER: 100.00%


T5 output saving

In [ ]:
T5_OUTPUT_FILE = Path(
    r"G:\Uni\Sem2\NLP\term_paper\data\t5_results.json"
)

output_data = {
    "model": MODEL_NAME,
    "top_k_results": {}
}

for top_k, results in all_t5_results.items():

    corpus_wer = wer(
        [r["reference"] for r in results],
        [r["corrected_text"] for r in results]
    )

    output_data["top_k_results"][str(top_k)] = {
        "corpus_wer": corpus_wer,
        "results": results
    }

with open(T5_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        output_data,
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"Saved T5 results to:\n{T5_OUTPUT_FILE}")